# Code Generator + Code Executor + 2 Tools
- Uses Langgraph

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "fkjdslf"

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


# Tools (2 normal + 1 code system)

In [2]:
from langchain_core.tools import tool

@tool
def calculator(expression: str) -> str:
    """Evaluate math expressions"""
    try:
        return str(eval(expression))
    except Exception as e:
        return str(e)

@tool
def generate_code(task: str) -> str:
    """Generate Python code for a given task"""
    
    prompt = f"""
    Write Python code ONLY for this task:
    {task}

    Rules:
    - No explanation
    - Only executable Python code
    """

    response = llm.invoke(prompt)
    return response.content


@tool
def run_python(code: str) -> str:
    """
    Execute Python code safely in a restricted environment and return its output.

    This tool runs dynamically generated Python code using `exec()` while capturing
    anything printed to stdout (e.g., via print statements). It uses a restricted
    execution context with no built-in functions to reduce security risks.

    Args:
        code (str): A string containing valid Python code to execute.

    Returns:
        str: 
            - The captured standard output produced by the code (e.g., print output), OR
            - An error message if execution fails.

    Example:
        Input:
            code = "print(5 * 5)"

        Output:
            "25\n"
    """

    import io
    import contextlib

    # SAFE built-ins
    safe_builtins = {
        "print": print,
        "range": range,
        "len": len,
        "sum": sum,
        "__import__": __import__,   # ✅ required for imports
    }

    try:
        output = io.StringIO()

        with contextlib.redirect_stdout(output):
            exec(code, {"__builtins__": safe_builtins}, {})

        return output.getvalue()

    except Exception as e:
        return f"Execution error: {str(e)}"


tools = {
    "calculator": calculator,
    "generate_code": generate_code,
    "run_python": run_python
}


# Router Node (decides tool)

In [3]:
from langchain_core.messages import SystemMessage

router_prompt = SystemMessage(content="""
You are a STRICT tool router.

Return ONLY JSON:

{
  "tool": "tool_name",
  "args": {...}
}

DECISION RULES:

1. Use "calculator" for:
   - math calculations
   - arithmetic (square, add, multiply, etc.)
   - numeric problems

2. Use "generate_code" ONLY when:
   - user explicitly asks for Python code
   - OR says "write code", "generate code", "execute code"


3. NEVER solve the problem yourself
4. NEVER explain

TOOLS:

- calculator → {"expression": "..."}
- generate_code → {"task": "..."}
- run_python → {"code": "..."}
""")


In [4]:
import json
import re

def parse_json(text):
    try:
        return json.loads(text)
    except:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if match:
            return json.loads(match.group())
    return {"tool": "none", "args": {}}


def router_node(state):
    response = llm.invoke([router_prompt] + state["messages"])

    decision = parse_json(response.content)

    return {
        **state,   # ✅ preserve previous state
        "messages": state["messages"] + [response],
        "tool_decision": decision
    }


# Tool Node (execution logic)

In [5]:
def clean_code(code):
    import re
    match = re.search(r"```python(.*?)```", code, re.DOTALL)
    return match.group(1).strip() if match else code.strip()

def tool_node(state):
    decision = state["tool_decision"]
    print("...ROUTER DECISION:", decision)  # debug

    tool_name = decision.get("tool")
    args = decision.get("args", {})

    # ✅ FIX 2 — add it RIGHT HERE
    if tool_name == "generate_code":
        if "task" not in args:
            # LLM may send {"code": "..."} or raw string
            args = {"task": args.get("code", str(args))}

    if tool_name == "run_python":
        if "code" not in args:
            args = {"code": str(args)}

    # ---------------------------------

    try:
        if tool_name == "none":
            result = "No tool used"

        elif tool_name == "calculator":
            result = calculator.invoke(args)

        elif tool_name == "generate_code":
            code = generate_code.invoke(args)

            clean = clean_code(code)

            execution = run_python.invoke({"code": clean})

            result = f"""
                CODE:
                {clean}
                
                OUTPUT:
                {execution}
            """

        else:
            result = "Unknown tool"

        return {
            **state,
            "messages": state["messages"] + [result],
            "retry_count": 0
        }

    except Exception as e:
        return {
            **state,
            "messages": state["messages"] + [str(e)],
            "retry_count": state.get("retry_count", 0) + 1
        }


# Retry Node

In [6]:
from typing import TypedDict, List, Dict, Any
from langchain_core.messages import BaseMessage

class AgentState(TypedDict):
    messages: List[BaseMessage]
    tool_decision: Dict[str, Any]   # ✅ MUST exist
    retry_count: int



def retry_node(state: AgentState):
    if state.get("retry_count", 0) > 2:
        return {
            "messages": state["messages"] + ["Failed after retries."]
        }
    return state


# Final Answer Node

In [7]:
def final_node(state: AgentState):
     return state


# Build LangGraph -> Flow -> Compile

In [8]:
from langgraph.graph import StateGraph, END

graph = StateGraph(AgentState)

# graph.add_node("router", router)
graph.add_node("router", router_node)
graph.add_node("tool", tool_node)
graph.add_node("retry", retry_node)
graph.add_node("final", final_node)



graph.set_entry_point("router")

graph.add_edge("router", "tool")
graph.add_edge("tool", "retry")

graph.add_conditional_edges(
    "retry",
    lambda s: "final" if s.get("retry_count", 0) < 2 else END
)

graph.add_edge("final", END)



app = graph.compile()

# Run Example (CODE EXECUTION)

```
Write python code to find factorial of 5 and execute it
Write Python code to add numbers from 1 to 10 and print the result
What is (45 + 55) * 2?
Find square of 19
What is capital of planet Mars ?


In [9]:
from langchain_core.messages import HumanMessage

result = app.invoke({
    "messages": [
        HumanMessage(content="Write python code to find factorial of 5 and execute it")
    ],
    "tool_decision": {},
    "retry_count": 0
})

print(result["messages"][-1])


...ROUTER DECISION: {'tool': 'generate_code', 'args': {'task': 'Write python code to find factorial of 5'}}

                CODE:
                import math

factorial_of_5 = math.factorial(5)
print(factorial_of_5)

                OUTPUT:
                120

            


In [10]:
from langchain_core.messages import HumanMessage

result = app.invoke({
    "messages": [
        HumanMessage(content="Write Python code to add numbers from 1 to 10 and print the result")
    ],
    "tool_decision": {},
    "retry_count": 0
})

print(result["messages"][-1])


...ROUTER DECISION: {'tool': 'generate_code', 'args': {'task': 'Write Python code to add numbers from 1 to 10 and print the result'}}

                CODE:
                result = sum(range(1, 11))
print(result)

                OUTPUT:
                55

            


In [11]:
from langchain_core.messages import HumanMessage

result = app.invoke({
    "messages": [
        HumanMessage(content="What is (45 + 55) * 2?")
    ],
    "tool_decision": {},
    "retry_count": 0
})

print(result["messages"][-1])


...ROUTER DECISION: {'tool': 'calculator', 'args': {'expression': '(45 + 55) * 2'}}
200


In [12]:
from langchain_core.messages import HumanMessage

result = app.invoke({
    "messages": [
        HumanMessage(content="Find square of 19")
    ],
    "tool_decision": {},
    "retry_count": 0
})

print(result["messages"][-1])


...ROUTER DECISION: {'tool': 'calculator', 'args': {'expression': '19 * 19'}}
361


In [13]:
from langchain_core.messages import HumanMessage

result = app.invoke({
    "messages": [
        HumanMessage(content="What is capital of planet Mars ?")
    ],
    "tool_decision": {},
    "retry_count": 0
})

print(result["messages"][-1])


...ROUTER DECISION: {'tool': 'tool_name', 'args': {}}
Unknown tool
